# Proyecto 4 — Detective: Demo del Sistema Multi-Agente

Este notebook demuestra paso a paso el sistema multi-agente del proyecto **detective**:

1. **Que es un sistema multi-agent** y por que importa
2. **Que es MCP** (Model Context Protocol)
3. **Los agentes** del sistema (triage, resolver, escalador, supervisor)
4. **Las herramientas** (kb_search, responder, history_lookup, escalate_human)
5. **La memoria** persistente en PostgreSQL
6. **El flujo completo** con LangGraph

## Que es un Sistema Multi-Agente

Un sistema multi-agente usa multiples agentes especializados que colaboran para resolver una tarea.
En vez de tener un solo LLM que hace todo, cada agente tiene una responsabilidad clara:

| Agente | Responsabilidad |
|--------|----------------|
| **Triage** | Clasificar la consulta del seller |
| **Resolver** | Ejecutar herramientas y generar respuesta |
| **Escalador** | Crear ticket cuando el resolver no puede |
| **Supervisor** | Orquestar el flujo completo |

## Que es MCP (Model Context Protocol)

MCP es un protocolo para exponer herramientas a agentes LLM. Cada herramienta es una funcion
que el agente puede invocar: buscar en la KB, consultar historial, escalar a humano, etc.

En nuestro sistema, las herramientas son:
- `kb_search` — busca en ChromaDB (knowledge base)
- `responder` — genera respuestas con el LLM
- `history_lookup` — consulta tickets anteriores del seller
- `escalate_human` — registra un ticket de escalamiento

## 0. Setup

Verificamos que el entorno este configurado y las dependencias instaladas.

In [ ]:
import sys
print(f"Python: {sys.version}")

# Verificar dependencias criticas
deps = ["langchain_core", "langchain_ollama", "langgraph", "psycopg2", "boto3", "chromadb"]
for dep in deps:
    try:
        __import__(dep)
        print(f"  {dep}: OK")
    except ImportError:
        print(f"  {dep}: FALTA — pip install {dep}")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Variables criticas
env_vars = {
    "OLLAMA_ENDPOINT": os.getenv("OLLAMA_ENDPOINT", "(no set)"),
    "MODEL_NAME": os.getenv("MODEL_NAME", "(no set)"),
    "POSTGRES_HOST": os.getenv("POSTGRES_HOST", "(no set)"),
    "MINIO_ENDPOINT": os.getenv("MINIO_ENDPOINT", "(no set)"),
}
for k, v in env_vars.items():
    print(f"  {k} = {v}")

## 1. Agente Triage — Clasificacion de Consultas

El agente de triaje clasifica cada consulta del seller en una categoria:
- **facturacion**: cobros, facturas, devoluciones de dinero
- **producto**: caracteristicas, disponibilidad, especificaciones
- **logistica**: envios, entregas, tracking
- **general**: cuentas, configuracion, quejas generales

Usa few-shot prompting con el LLM para obtener una clasificacion estructurada.

In [ ]:
from agents.triage import TriageAgent

triage = TriageAgent()

# Escenario 1: consulta de facturacion
result = triage.classify("Me cobraron el doble en mi factura del mes pasado")
print(f"Categoria: {result['categoria']}")
print(f"Confianza: {result['confianza']}")
print(f"Razon: {result['razon']}")

In [ ]:
# Escenario 2: consulta de producto
result2 = triage.classify("El producto que recibi no tiene las caracteristicas que prometia el anuncio")
print(f"Categoria: {result2['categoria']} | Confianza: {result2['confianza']}")

In [ ]:
# Escenario 3: consulta de logistica
result3 = triage.classify("Mi pedido lleva 2 semanas sin llegar y no hay tracking actualizado")
print(f"Categoria: {result3['categoria']} | Confianza: {result3['confianza']}")

In [ ]:
# Escenario 4: consulta ambigua (baja confianza)
result4 = triage.classify("Algo esta mal con mi cuenta pero no se exactamente que")
print(f"Categoria: {result4['categoria']} | Confianza: {result4['confianza']}")
print(f"Esto deberia escalar si la confianza es < {os.getenv('CONFIDENCE_THRESHOLD', '0.6')}")

## 2. Herramientas MCP — kb_search y responder

Las herramientas son funciones que el agente puede invocar:

- **kb_search**: busca en ChromaDB los chunks mas relevantes para una consulta
- **responder**: genera una respuesta usando el contexto recuperado

Si ChromaDB o la API de helmet no estan disponibles, las tools usan mocks para testing.

In [ ]:
from tools.kb_search import kb_search
from tools.responder import responder

# Buscar en la KB
chunks = kb_search("problemas con facturacion", k=3)
print(f"Chunks encontrados: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] score={chunk['score']:.2f} | {chunk['content'][:80]}...")

In [ ]:
# Generar respuesta con contexto
contexto = "\n".join(c["content"] for c in chunks)
answer, tokens = responder("Me cobraron el doble", context=contexto)
print(f"Respuesta ({tokens} tokens):")
print(answer[:500])

## 3. Herramienta MCP — history_lookup

Consulta el historial de tickets anteriores del seller en PostgreSQL.
Permite al agente resolver saber si el seller ya tuvo problemas similares.

In [ ]:
from tools.history_lookup import history_lookup

tickets = history_lookup("seller_001", limit=5)
print(f"Tickets encontrados: {len(tickets)}")
for t in tickets:
    print(f"  [{t['status']}] {t['subject']}")

## 4. Agente Resolver — Resolucion con Herramientas

El resolver ejecuta las herramientas en secuencia:
1. `kb_search` para contexto de la knowledge base
2. `responder` para generar una respuesta base
3. `history_lookup` para verificar historial del seller
4. LLM para refinar la respuesta final con citas

In [ ]:
from agents.resolver import ResolverAgent

resolver = ResolverAgent()

result = resolver.resolve(
    query="Me cobraron el doble en mi factura del mes pasado",
    categoria="facturacion",
    seller_id="seller_001",
)

print("Respuesta:")
print(result["respuesta"][:500])
print(f"\nCitas: {result['citas']}")
print(f"Necesita escalamiento: {result['necesita_escalamiento']}")

## 5. Agente Escalador — Tickets de Escalamiento

Cuando el resolver no puede atender la consulta o la confianza es baja,
el escalador genera un ticket con todo el contexto para un agente humano.

In [ ]:
from agents.escalador import EscaladorAgent

escalador = EscaladorAgent()

ticket = escalador.escalate(
    query="Mi cuenta fue bloqueada sin razon alguna y necesito acceso urgente",
    categoria="general",
    razon_escalamiento="El seller reporta bloqueo de cuenta sin explicacion. Requiere revision humana.",
    seller_id="seller_002",
)

print(f"Ticket ID: {ticket['ticket_id']}")
print(f"Asunto: {ticket['ticket_data']['subject']}")
print(f"Prioridad: {ticket['ticket_data']['priority']}")

## 6. Supervisor — Flujo Completo con LangGraph

El supervisor orquesta todo el flujo usando **LangGraph** (StateGraph):

```
triage → resolver → (condicional) → escalador → format_answer → END
                                  → format_answer → END
```

El estado se comparte entre nodos via `AgentState` (TypedDict).
La memoria de conversacion se persiste en PostgreSQL.

In [ ]:
from agents.supervisor import SupervisorAgent

supervisor = SupervisorAgent()

# Escenario 1: consulta de facturacion (deberia resolver sin escalar)
result1 = supervisor.run(
    query="Me cobraron el doble en mi factura del mes pasado",
    seller_id="seller_100",
)
print("=== Escenario 1: Facturacion ===")
print(f"Categoria: {result1['categoria']}")
print(f"Confianza: {result1['confianza_triaje']}")
print(f"Ticket: {result1['ticket_id'] or 'N/A (resuelto automaticamente)'}")
print(f"Respuesta: {result1['final_output'][:300]}")

In [ ]:
# Escenario 2: consulta que necesita KB (producto)
result2 = supervisor.run(
    query="Cuales son las caracteristicas delProducto XYZ? Necesito saber si es compatible con mi setup",
    seller_id="seller_101",
)
print("=== Escenario 2: Producto (KB) ===")
print(f"Categoria: {result2['categoria']}")
print(f"Respuesta: {result2['final_output'][:300]}")

In [ ]:
# Escenario 3: consulta que necesita historial
result3 = supervisor.run(
    query="Otra vez tengo el mismo problema de envio que la vez anterior",
    seller_id="seller_001",
)
print("=== Escenario 3: Logistica (historial) ===")
print(f"Categoria: {result3['categoria']}")
print(f"Respuesta: {result3['final_output'][:300]}")

In [ ]:
# Escenario 4: consulta que se escala
result4 = supervisor.run(
    query="Mi cuenta fue bloqueada arbitrariamente y necesito acceso inmediato para gestionar mis pedidos pendientes",
    seller_id="seller_002",
)
print("=== Escenario 4: Escalamiento ===")
print(f"Categoria: {result4['categoria']}")
print(f"Ticket ID: {result4['ticket_id']}")
print(f"Respuesta: {result4['final_output'][:300]}")

## 7. Memoria de Conversacion

El supervisor guarda automaticamente cada interaccion en PostgreSQL.
Podemos recuperar el historial completo de un seller.

In [ ]:
# Ver historial de seller_100
history = supervisor.get_history("seller_100")
print(f"Mensajes en historial de seller_100: {len(history)}")
for msg in history:
    print(f"  [{msg['role']}] {msg['content'][:100]}...")

## 8. Resumen del Flujo

El sistema multi-agente funciona asi:

1. El seller envia una consulta
2. **Triage** la clasifica (categoria + confianza)
3. **Resolver** ejecuta las herramientas MCP y genera una respuesta
4. Si la confianza es baja o el resolver no puede → **Escalador** genera un ticket
5. El **Supervisor** orquesta todo via LangGraph y persiste la memoria

### Que aprendemos

- **Multi-agente**: dividir responsabilidades entre agentes especializados
- **MCP**: exponer herramientas como funciones invocables
- **LangGraph**: orquestar flujos complejos con grafo de estado
- **Memoria**: persistir conversaciones en PostgreSQL
- **Escalamiento**: saber cuando delegar a un humano